## Bước 9: Hyperparameter Tuning
### 9.1 Đọc dữ liệu và import thư viện

## Thiết lập khả năng tái lập

Seed được đặt đồng nhất cho Python, NumPy và PyTorch. Các script mới còn sử dụng generator có seed cho DataLoader.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from common import set_global_seed

SEED = 42
set_global_seed(SEED)
print(f"Đã thiết lập seed tái lập: {SEED}")


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score
from xgboost import XGBClassifier

df = pd.read_csv('../data/processed/depression_severity_preprocessed.csv')
label_map = {'minimum': 0, 'mild': 1, 'moderate': 2, 'severe': 3}
df['label_num'] = df['label'].map(label_map)

X_text = df['text_classical'].values
y = df['label_num'].values

print("Số dòng:", len(df))
print("Phân bố nhãn:", np.bincount(y))

Số dòng: 3519
Phân bố nhãn: [2555  290  393  281]


### 9.2 Tune Logistic Regression

In [3]:
from sklearn.model_selection import train_test_split

# Chia riêng 1 lần để tìm tham số (tách biệt với 5-fold đánh giá cuối)
X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
    X_text, y, test_size=0.2, stratify=y, random_state=42
)

vectorizer_tune = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2, max_df=0.9)
X_train_vec = vectorizer_tune.fit_transform(X_train_tune)
X_val_vec = vectorizer_tune.transform(X_val_tune)

param_grid_logreg = {
    'C': [0.1, 0.5, 1.0, 5.0, 10.0],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

macro_f1_scorer = make_scorer(f1_score, average='macro')

grid_logreg = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    param_grid_logreg,
    scoring=macro_f1_scorer,
    cv=3
)
grid_logreg.fit(X_train_vec, y_train_tune)

print("Tham số tốt nhất:", grid_logreg.best_params_)
print("Macro-F1 tốt nhất (cross-val trên tập train):", grid_logreg.best_score_.round(4))

val_pred = grid_logreg.predict(X_val_vec)
val_f1 = f1_score(y_val_tune, val_pred, average='macro')
print("Macro-F1 trên tập validation (chưa từng thấy):", round(val_f1, 4))

c:\Users\Thanh Hue\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Thanh Hue\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Thanh Hue\miniconda3\Lib\site-packages\sk

Tham số tốt nhất: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}
Macro-F1 tốt nhất (cross-val trên tập train): 0.4141
Macro-F1 trên tập validation (chưa từng thấy): 0.4601


c:\Users\Thanh Hue\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


### 9.2b Tắt cảnh báo deprecation (chỉ để output gọn hơn)

In [4]:
import warnings
warnings.filterwarnings('ignore')

### 9.3 Tune SVM

In [5]:
param_grid_svm = {
    'C': [0.1, 0.5, 1.0, 5.0],
    'kernel': ['linear']
}

grid_svm = GridSearchCV(
    SVC(class_weight='balanced', random_state=42),
    param_grid_svm,
    scoring=macro_f1_scorer,
    cv=3
)
grid_svm.fit(X_train_vec, y_train_tune)

print("Tham số tốt nhất:", grid_svm.best_params_)
print("Macro-F1 tốt nhất (cross-val trên tập train):", round(grid_svm.best_score_, 4))

val_pred_svm = grid_svm.predict(X_val_vec)
val_f1_svm = f1_score(y_val_tune, val_pred_svm, average='macro')
print("Macro-F1 trên tập validation:", round(val_f1_svm, 4))

Tham số tốt nhất: {'C': 0.5, 'kernel': 'linear'}
Macro-F1 tốt nhất (cross-val trên tập train): 0.405
Macro-F1 trên tập validation: 0.458


### 9.4 Tune XGBoost (Random Search)

In [7]:
param_dist_xgb = {
    'n_estimators': [100, 150],
    'max_depth': [4, 6],
    'learning_rate': [0.1, 0.2],
}

random_xgb = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='mlogloss', n_jobs=1),
    param_dist_xgb,
    n_iter=4,
    scoring=macro_f1_scorer,
    cv=2,
    random_state=42,
    n_jobs=1,
    verbose=2
)
random_xgb.fit(X_train_xgb_tune, y_train_tune, sample_weight=sample_weights_tune)

print("Tham số tốt nhất:", random_xgb.best_params_)
print("Macro-F1 tốt nhất (cross-val trên tập train):", round(random_xgb.best_score_, 4))

val_pred_xgb = random_xgb.predict(X_val_xgb_tune)
val_f1_xgb = f1_score(y_val_tune, val_pred_xgb, average='macro')
print("Macro-F1 trên tập validation:", round(val_f1_xgb, 4))

Fitting 2 folds for each of 4 candidates, totalling 8 fits
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=150; total time=  11.8s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=150; total time=  13.0s
[CV] END ...learning_rate=0.2, max_depth=4, n_estimators=150; total time=  11.2s
[CV] END ...learning_rate=0.2, max_depth=4, n_estimators=150; total time=  27.1s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=100; total time=  25.4s
[CV] END ...learning_rate=0.1, max_depth=4, n_estimators=100; total time=  22.9s
[CV] END ...learning_rate=0.2, max_depth=6, n_estimators=150; total time=  41.8s
[CV] END ...learning_rate=0.2, max_depth=6, n_estimators=150; total time=  43.1s
Tham số tốt nhất: {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1}
Macro-F1 tốt nhất (cross-val trên tập train): 0.2576
Macro-F1 trên tập validation: 0.4475


## Tuning tối thiểu cho DistilBERT

Quy trình mới giữ riêng tập test, lựa chọn cấu hình trên validation theo Macro-F1 và QWK, sau đó mới đánh giá cấu hình tốt nhất trên test. Không sử dụng test để chọn learning rate, epoch hoặc max length.

In [ ]:
RUN_DISTILBERT_TUNING = False  # Có thể mất nhiều giờ trên CPU

if RUN_DISTILBERT_TUNING:
    import subprocess
    import sys
    from pathlib import Path

    PROJECT_ROOT = Path('..').resolve()
    subprocess.run(
        [sys.executable, 'src/tune_distilbert.py', '--max-configs', '4'],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print('Đặt RUN_DISTILBERT_TUNING=True để thử tối thiểu 4 cấu hình DistilBERT.')


In [ ]:
from pathlib import Path
import pandas as pd

path = Path('../outputs/reports/distilbert_tuning_results.csv')
if path.exists():
    display(pd.read_csv(path).sort_values(['macro_f1', 'qwk'], ascending=False))
else:
    print('Chưa có kết quả tuning DistilBERT.')
